<a href="https://colab.research.google.com/github/Durvankur-Rajam/Impactsure_Project/blob/main/Week8_Task14%2C15.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install langchain langchain-community chromadb sentence-transformers pymupdf transformers accelerate -q
print("Done")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 46.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 38.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 39.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 51.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 23.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 52.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 k

In [2]:
import fitz
import re
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from sentence_transformers import SentenceTransformer
import chromadb
print("Imports done")

Imports done


In [3]:
!wget -q "https://arxiv.org/pdf/2109.07958" -O truthfulqa.pdf
!wget -q "https://arxiv.org/pdf/2303.08896" -O selfcheckgpt.pdf
!wget -q "https://arxiv.org/pdf/1706.03762" -O attention.pdf
print("PDFs downloaded")

PDFs downloaded


In [4]:
def extract_text(pdf_path):
    doc = fitz.open(pdf_path)
    text = ""
    for page in doc:
        text += page.get_text()
    doc.close()
    text = re.sub(r'\n{3,}', '\n\n', text)
    text = re.sub(r'[ \t]+', ' ', text)
    return text.strip()

def chunk_text(text, chunk_size=500, overlap=50):
    words = text.split()
    chunks = []
    i = 0
    while i < len(words):
        chunk = " ".join(words[i:i+chunk_size])
        chunks.append(chunk)
        i += chunk_size - overlap
    return chunks

pdfs = {
    "TruthfulQA"  : "truthfulqa.pdf",
    "SelfCheckGPT": "selfcheckgpt.pdf",
    "Attention"   : "attention.pdf"
}

all_chunks = []
all_ids    = []
all_metas  = []

for name, path in pdfs.items():
    text   = extract_text(path)
    chunks = chunk_text(text)
    for i, chunk in enumerate(chunks):
        all_chunks.append(chunk)
        all_ids.append(f"{name}_{i}")
        all_metas.append({"source": name})
    print(f"{name}: {len(chunks)} chunks")

print(f"\nTotal chunks: {len(all_chunks)}")

TruthfulQA: 33 chunks
SelfCheckGPT: 19 chunks
Attention: 14 chunks

Total chunks: 66


In [5]:
embedder = SentenceTransformer("all-MiniLM-L6-v2")
print("Embedder loaded")

client     = chromadb.Client()
collection = client.create_collection("pdf_knowledge")

batch_size = 100
for i in range(0, len(all_chunks), batch_size):
    batch_chunks = all_chunks[i:i+batch_size]
    batch_ids    = all_ids[i:i+batch_size]
    batch_metas  = all_metas[i:i+batch_size]
    embeddings   = embedder.encode(batch_chunks).tolist()
    collection.add(
        documents  = batch_chunks,
        embeddings = embeddings,
        ids        = batch_ids,
        metadatas  = batch_metas
    )
    print(f"Stored batch {i//batch_size + 1}")

print(f"\nTotal documents in ChromaDB: {collection.count()}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedder loaded
Stored batch 1

Total documents in ChromaDB: 66


In [6]:
def retrieve(query, top_k=3):
    query_embedding = embedder.encode([query]).tolist()
    results = collection.query(
        query_embeddings = query_embedding,
        n_results        = top_k
    )
    return results['documents'][0], results['metadatas'][0]

query = "What is the attention mechanism in transformers?"
docs, metas = retrieve(query)
print(f"Query: {query}\n")
for i, (doc, meta) in enumerate(zip(docs, metas)):
    print(f"Result {i+1} [{meta['source']}]:")
    print(f"{doc[:200]}...")
    print()

Query: What is the attention mechanism in transformers?

Result 1 [Attention]:
work we employ h = 8 parallel attention layers, or heads. For each of these we use dk = dv = dmodel/h = 64. Due to the reduced dimension of each head, the total computational cost is similar to that o...

Result 2 [Attention]:
representations of its input and output without using sequence- aligned RNNs or convolution. In the following sections, we will describe the Transformer, motivate self-attention and discuss its advant...

Result 3 [Attention]:
and encoder-decoder architectures [38, 24, 15]. Recurrent models typically factor computation along the symbol positions of the input and output sequences. Aligning the positions to steps in computati...



In [7]:
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tokenizer  = AutoTokenizer.from_pretrained(model_name)
model      = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype  = torch.float16,
    device_map   = "auto"
)
print(f"Model loaded on: {next(model.parameters()).device}")

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Model loaded on: cuda:0


In [8]:
def rag_answer(question, max_new_tokens=150):
    # Step 1: Retrieve relevant chunks
    docs, metas = retrieve(question, top_k=3)
    context     = "\n\n".join(docs)

    # Step 2: Build augmented prompt
    prompt = f"""<|system|>
You are a factual assistant. Use the context below to answer the question accurately.</s>
<|user|>
Context:
{context[:1500]}

Question: {question}</s>
<|assistant|>"""

    # Step 3: Generate answer
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens   = max_new_tokens,
            temperature      = 0.7,
            do_sample        = True,
            pad_token_id     = tokenizer.eos_token_id
        )
    full_output = tokenizer.decode(outputs[0], skip_special_tokens=True)
    answer      = full_output.split("<|assistant|>")[-1].strip()
    return answer, docs, metas

In [9]:
test_questions = [
    "What is TruthfulQA and why was it created?",
    "How does SelfCheckGPT detect hallucinations?",
    "What is the attention mechanism in transformers?",
    "What is the inverse scaling problem in LLMs?",
    "How does multi-head attention work?"
]

print("Running RAG pipeline on 5 test questions...")
print("="*60)

for q in test_questions:
    answer, docs, metas = rag_answer(q)
    sources = list(set([m['source'] for m in metas]))
    print(f"\nQ: {q}")
    print(f"Sources used: {sources}")
    print(f"A: {answer[:200]}...")
    print("-"*60)

Running RAG pipeline on 5 test questions...

Q: What is TruthfulQA and why was it created?
Sources used: ['TruthfulQA']
A: The question is about TruthfulQA, a test set of 817 questions designed for the zero-shot setting and intended only for the author's purpose of testing for a weakness in the truthfulness of language mo...
------------------------------------------------------------

Q: How does SelfCheckGPT detect hallucinations?
Sources used: ['SelfCheckGPT']
A: SelfCheckGPT is designed to detect hallucinations by comparing multiple sampled responses and measuring consistency. Notation: Let R refer to an LLM response drawn from a given user query. SelfCheckGP...
------------------------------------------------------------

Q: What is the attention mechanism in transformers?
Sources used: ['Attention']
A: The attention mechanism in transformers is multi-head attention, which is used in three different ways:

1. Encoder-decoder attention: In "encoder-decoder attention" layers, the qu

In [10]:
import pandas as pd

rag_results = []
for q in test_questions:
    answer, docs, metas = rag_answer(q)
    sources = list(set([m['source'] for m in metas]))
    rag_results.append({
        "question"      : q,
        "rag_answer"    : answer,
        "sources_used"  : str(sources),
        "context_length": len("\n\n".join(docs))
    })

df_rag = pd.DataFrame(rag_results)
df_rag.to_csv("rag_results.csv", index=False)
print("Saved to rag_results.csv")
print(f"\nRAG pipeline working on {len(df_rag)} questions")

Saved to rag_results.csv

RAG pipeline working on 5 questions


In [11]:
!pip install selfcheckgpt -q
print("Done")

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 2.8 MB/s eta 0:00:00
Done


In [12]:
from selfcheckgpt.modeling_selfcheck import SelfCheckBERTScore
import torch
print("SelfCheckGPT imported")

SelfCheckGPT imported


In [13]:
def generate_multiple_samples(question, n_samples=5, max_new_tokens=100):
    prompt = f"""<|system|>
You are a factual assistant. Answer the question truthfully and concisely.</s>
<|user|>
{question}</s>
<|assistant|>"""

    samples = []
    inputs  = tokenizer(prompt, return_tensors="pt").to("cuda")

    for i in range(n_samples):
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens = max_new_tokens,
                temperature    = 0.7,
                do_sample      = True,
                pad_token_id   = tokenizer.eos_token_id
            )
        full_output = tokenizer.decode(outputs[0], skip_special_tokens=True)
        answer      = full_output.split("<|assistant|>")[-1].strip()
        samples.append(answer)

    return samples

test_q   = test_questions[0]
samples  = generate_multiple_samples(test_q, n_samples=3)
print(f"Q: {test_q}\n")
for i, s in enumerate(samples):
    print(f"Sample {i+1}: {s[:150]}...")

Q: What is TruthfulQA and why was it created?

Sample 1: TruthfulQA is a free, open-source Q&A platform platform that uses natural language processing (NLP) to generate question and answer pairs from a datab...
Sample 2: TruthfulQA is a chatbot platform designed to create factual assistant chatbots that provide users with accurate, reliable, and trustworthy information...
Sample 3: TruthfulQA is a Python library for building and training questions-answering models. The library is designed to make it easy for researchers, academic...


In [16]:
selfcheck = SelfCheckBERTScore(rescale_with_baseline=True)
print("SelfCheckBERTScore loaded")

selfcheck_results = []

for q in test_questions:
    print(f"\nProcessing: {q[:50]}...")

    samples      = generate_multiple_samples(q, n_samples=3)
    main_answer  = samples[0]
    other_samples = samples[1:]

    sentences = [s.strip() for s in main_answer.split('.') if len(s.strip()) > 10]

    if not sentences:
        print("No sentences found, skipping")
        continue

    scores = selfcheck.predict(
        sentences        = sentences,
        sampled_passages = other_samples
    )

    scores_list = scores.tolist() if hasattr(scores, 'tolist') else list(scores)

    avg_score = sum(scores_list) / len(scores_list) if scores_list else 0

    selfcheck_results.append({
        "question"            : q,
        "main_answer"         : main_answer,
        "avg_selfcheck_score" : round(avg_score, 3),
        "sentence_scores"     : str([round(s, 3) for s in scores_list]),
        "hallucination_flag"  : avg_score > 0.5
    })

    print(f"Avg SelfCheck Score : {avg_score:.3f}")
    print(f"Hallucination flag  : {avg_score > 0.5}")

SelfCheck-BERTScore initialized
SelfCheckBERTScore loaded

Processing: What is TruthfulQA and why was it created?...


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

Avg SelfCheck Score : 0.694
Hallucination flag  : True

Processing: How does SelfCheckGPT detect hallucinations?...


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

Avg SelfCheck Score : 0.678
Hallucination flag  : True

Processing: How does multi-head attention work?...


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

Avg SelfCheck Score : 0.610
Hallucination flag  : True


In [18]:
df_selfcheck = pd.DataFrame(selfcheck_results)
df_selfcheck.to_csv("selfcheck_scores.csv", index=False)

print("="*60)
print("SELFCHECKGPT RESULTS")
print("="*60)
print(df_selfcheck[['question', 'avg_selfcheck_score', 'hallucination_flag']].to_string())
print("\nNote: Higher score = more inconsistency = more likely hallucinated")
print(f"\nFlagged as hallucinated: {df_selfcheck['hallucination_flag'].sum()}/{len(df_selfcheck)}")
print("\nSaved to selfcheck_scores.csv")

SELFCHECKGPT RESULTS
                                           question  avg_selfcheck_score  hallucination_flag
0        What is TruthfulQA and why was it created?                0.694                True
1      How does SelfCheckGPT detect hallucinations?                0.608                True
2  What is the attention mechanism in transformers?                0.652                True
3      What is the inverse scaling problem in LLMs?                0.678                True
4               How does multi-head attention work?                0.610                True

Note: Higher score = more inconsistency = more likely hallucinated

Flagged as hallucinated: 5/5

Saved to selfcheck_scores.csv
